Code that will identify follicular gradients and quantify the percentage of inner and outer follicle cells as you move through the transition zone. Use this code after clustering probability bins in CleanHierarchicalFollicle.

Read in your packages.

In [2]:
#Import Packages
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import norm

import time
import sys
import matplotlib.pyplot as plt
import math
import os

from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

Read in the dataframe containing all the information with all the probabilities for each cell for each neighborhood.

In [ ]:
df = pd.read_csv(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\SO_Project2\intestine_all_information_2.csv')
df

Read in the dataframe containing the information about the cluster assignment from probability bin clustering.

In [ ]:
total_df = pd.read_csv(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\SO_Project2\intestine_all_probabilitybincluster.csv')

total_df

Plotting for paper figures showing B006_Descending-Sigmoid probability bin clusters.

In [4]:
#Import Neighborhood Functions and Code
cellhier_path = r"C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\cellhier"
sys.path.append(cellhier_path)
from general import *
from plot_john_2 import *

In [ ]:
#Subset of clusters to investigate/plot
neigh_list = [0,1,2,3,4,5,6,7,8,9]

#modify figure size aesthetics for each neighborhood
plt.rcParams["legend.markerscale"] = 15
figs = catplot2(total_df.loc[total_df['unique_region']== 'B006_Descending - Sigmoid'], X = 'x', Y='y', exp = 'unique_region',
               hue = 'Probability_Bin_Cluster', axis='off', invert_y=False, size = 1, figsize=10, legend=False, title_fontsize=35)

A gradient of banded clusters was observed around the follicle structures as you move from innermost to outermost follicle, follows the gradient observed in the heatmap for degree of enrichment in probability bins.

Plot only the "high" probability bin clusters on spatial coordinates in B006_Descending-Sigmoid.

In [ ]:
#Subset of clusters to investigate/plot
neigh_list = [3,1,6,8,5]
filtered_df = total_df.loc[total_df['Probability_Bin_Cluster'].isin(neigh_list)]

#modify figure size aesthetics for each neighborhood
plt.rcParams["legend.markerscale"] = 15
figs = catplot2(filtered_df.loc[filtered_df['unique_region']== 'B006_Descending - Sigmoid'], X = 'x', Y='y', exp = 'unique_region',
               hue = 'Probability_Bin_Cluster', axis='off', invert_y=False, size = 1, figsize=10, legend=False, title_fontsize=35)

Investigate the spatial arrangement of each of the clusters in one unique region. Plot each cluster on a separate plot, color by the cell's assigned probility.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Filter the data for the specific unique region "B004_Ascending"
region_data = total_df[total_df['unique_region'] == 'B004_Ascending']

# List of all probability bins to plot
probability_bins = region_data['Probability_Bin_Cluster'].unique()

# Create a 2 by 5 grid layout for plotting
n_cols = 5  # Number of columns
n_rows = 2  # Number of rows (adjust based on the number of unique probability bins)
fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(n_cols*6, n_rows*6))  # 2x5 grid

# Flatten axes array for easier indexing
axes = axes.flatten()

# Loop over probability bins and plot
for j, prob_bin in enumerate(probability_bins):
    # Filter data for the current probability bin
    prob_bin_data = region_data[region_data['Probability_Bin_Cluster'] == prob_bin]
    
    # Plotting on the corresponding subplot
    ax = axes[j]  # Choose the axis for this probability bin
    scatter = ax.scatter(prob_bin_data['x'], prob_bin_data['y'], 
                         c=prob_bin_data['Assigned_Probability'], cmap='viridis', s=5, alpha=0.6)
    
    # Add title, labels, and adjust axis
    ax.set_title(f'Cluster {prob_bin} - B004_Ascending')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(True)
    
    # Add colorbar for the assigned probability
    fig.colorbar(scatter, ax=ax, label='Assigned Probability')

# Adjust layout to avoid overlap
plt.tight_layout()
plt.show()


Plot zoom in of B006_Descending-Sigmoid follicle colored by probability bin cluster.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Define your target settings
target_region = 'B006_Descending - Sigmoid'

# Filter for the specific region of interest
region_data = total_df[total_df['unique_region'] == target_region]

# Filter for the x and y range
region_data_filtered = region_data[
    (region_data['x'] >= 3000) & (region_data['x'] <= 5000) &
    (region_data['y'] >= 2500) & (region_data['y'] <= 4500)
]

# Get unique Probability_Bin_Cluster values
unique_probs = sorted(region_data_filtered['Probability_Bin_Cluster'].dropna().unique())

# Define the same color palette as used in catplot2 (e.g., "bright")
palette = sns.color_palette("bright", len(unique_probs))

# Map colors to Probability_Bin_Cluster using the same palette
color_map = {prob: palette[i] for i, prob in enumerate(unique_probs)}

# -------- MAIN PLOT (WITHOUT LEGEND) -------- #
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

# Scatter plot with color map based on Probability_Bin_Cluster
scatter = ax.scatter(
    region_data_filtered['x'],
    region_data_filtered['y'],
    c=region_data_filtered['Probability_Bin_Cluster'].map(color_map),
    s=2, alpha=1
)

# Turn off axes and set title
ax.axis('off')
ax.set_title(f'{target_region} by Probability Bin Cluster', fontsize=25, pad=20)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# -------- LEGEND FIGURE ONLY -------- #
fig_legend = plt.figure(figsize=(1.5, 8), dpi=300)

# Create custom legend elements
legend_elements = [
    Patch(facecolor=color_map[prob], label=f'Cluster {prob}') for prob in unique_probs
]

# Add legend to the separate figure
fig_legend.legend(
    handles=legend_elements,
    title='Probability Bin Cluster',
    loc='center',
    ncol=1,
    frameon=True,
    fontsize=35,
    title_fontsize=35
)

# Remove axes from the legend figure
fig_legend.subplots_adjust(left=0, right=1, top=1, bottom=0)  # Adjust layout
fig_legend.gca().axis('off')  # Ensure no axes are present

# Save and display the figure
plt.savefig("legend_only.png", bbox_inches='tight')  # Optional: save
plt.show()


Plot "High" cells in the gradient that was observed. Also quantify the number of inner vs outer follicle cells as you move out the "gradient". Zoom in on the follicle in B006_Descending-Sigmoid.

In [ ]:
# Define your filter
target_prob = 3
neigh_inner = 'Inner Follicle'
neigh_outer = 'Outer Follicle'
target_region = 'B006_Descending - Sigmoid'

# Filter for the specific region of interest
region_data = total_df[total_df['unique_region'] == target_region]

# Filter for the x and y range
region_data_filtered = region_data[(region_data['x'] >= 3000) & (region_data['x'] <= 5000) & 
                                   (region_data['y'] >= 2500) & (region_data['y'] <= 4500)]

# Masks for matching cells in Inner and Outer Follicle neighborhoods
mask_inner = (
    (region_data_filtered['Neighborhood'] == neigh_inner) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)
mask_outer = (
    (region_data_filtered['Neighborhood'] == neigh_outer) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)

# Calculate the total number of cells in Inner and Outer Follicle neighborhoods
total_cells_in_cluster = len(region_data_filtered[mask_inner | mask_outer])

# Calculate the number of Inner and Outer Follicle cells in the cluster
inner_cells = len(region_data_filtered[mask_inner])
outer_cells = len(region_data_filtered[mask_outer])

# Calculate percentages of Inner and Outer Follicle cells within the cluster
prop_inner = inner_cells / total_cells_in_cluster * 100
prop_outer = outer_cells / total_cells_in_cluster * 100

# Plotting
plt.figure(figsize=(8, 8), dpi=300)

# Plot other cells in light gray
plt.scatter(
    region_data_filtered.loc[~(mask_inner | mask_outer), 'x'],
    region_data_filtered.loc[~(mask_inner | mask_outer), 'y'],
    c='lightgray', s=2, alpha=0.3, label='Other Cells'
)

# Plot Inner Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_inner, 'x'],
    region_data_filtered.loc[mask_inner, 'y'],
    c='teal', s=2, alpha=0.8, label=f'Inner Follicle ({prop_inner:.2f}%)'
)

# Plot Outer Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_outer, 'x'],
    region_data_filtered.loc[mask_outer, 'y'],
    c='orange', s=2, alpha=0.8, label=f'Outer Follicle ({prop_outer:.2f}%)'
)

# Remove axes
plt.axis('off')

# Title styling
plt.title(f'Cluster {target_prob} in Inner/Outer Follicle\n{target_region}',
          fontsize=25, pad=20)

# Legend styling
plt.legend(
    fontsize=20,
    title_fontsize=20,
    markerscale=10
)

plt.tight_layout()
plt.show()

In [ ]:
# Define your filter
target_prob = 6
neigh_inner = 'Inner Follicle'
neigh_outer = 'Outer Follicle'
target_region = 'B006_Descending - Sigmoid'

# Filter for the specific region of interest
region_data = total_df[total_df['unique_region'] == target_region]

# Filter for the x and y range
region_data_filtered = region_data[(region_data['x'] >= 3000) & (region_data['x'] <= 5000) & 
                                   (region_data['y'] >= 2500) & (region_data['y'] <= 4500)]

# Masks for matching cells in Inner and Outer Follicle neighborhoods
mask_inner = (
    (region_data_filtered['Neighborhood'] == neigh_inner) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)
mask_outer = (
    (region_data_filtered['Neighborhood'] == neigh_outer) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)

# Calculate the total number of cells in Inner and Outer Follicle neighborhoods
total_cells_in_cluster = len(region_data_filtered[mask_inner | mask_outer])

# Calculate the number of Inner and Outer Follicle cells in the cluster
inner_cells = len(region_data_filtered[mask_inner])
outer_cells = len(region_data_filtered[mask_outer])

# Calculate percentages of Inner and Outer Follicle cells within the cluster
prop_inner = inner_cells / total_cells_in_cluster * 100
prop_outer = outer_cells / total_cells_in_cluster * 100


# Plotting
plt.figure(figsize=(8, 8), dpi=300)

# Plot other cells in light gray
plt.scatter(
    region_data_filtered.loc[~(mask_inner | mask_outer), 'x'],
    region_data_filtered.loc[~(mask_inner | mask_outer), 'y'],
    c='lightgray', s=2, alpha=0.3, label='Other Cells'
)

# Plot Inner Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_inner, 'x'],
    region_data_filtered.loc[mask_inner, 'y'],
    c='teal', s=2, alpha=0.8, label=f'Inner Follicle ({prop_inner:.2f}%)'
)

# Plot Outer Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_outer, 'x'],
    region_data_filtered.loc[mask_outer, 'y'],
    c='orange', s=2, alpha=0.8, label=f'Outer Follicle ({prop_outer:.2f}%)'
)

# Remove axes
plt.axis('off')

# Title styling
plt.title(f'Cluster {target_prob} in Inner/Outer Follicle\n{target_region}',
          fontsize=25, pad=20)

# Legend styling
plt.legend(
    fontsize=20,
    title_fontsize=20,
    markerscale=10
)

plt.tight_layout()
plt.show()

In [ ]:
# Define your filter
target_prob = 1
neigh_inner = 'Inner Follicle'
neigh_outer = 'Outer Follicle'
target_region = 'B006_Descending - Sigmoid'

# Filter for the specific region of interest
region_data = total_df[total_df['unique_region'] == target_region]

# Filter for the x and y range
region_data_filtered = region_data[(region_data['x'] >= 3000) & (region_data['x'] <= 5000) & 
                                   (region_data['y'] >= 2500) & (region_data['y'] <= 4500)]

# Masks for matching cells in Inner and Outer Follicle neighborhoods
mask_inner = (
    (region_data_filtered['Neighborhood'] == neigh_inner) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)
mask_outer = (
    (region_data_filtered['Neighborhood'] == neigh_outer) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)

# Calculate the total number of cells in Inner and Outer Follicle neighborhoods
total_cells_in_cluster = len(region_data_filtered[mask_inner | mask_outer])

# Calculate the number of Inner and Outer Follicle cells in the cluster
inner_cells = len(region_data_filtered[mask_inner])
outer_cells = len(region_data_filtered[mask_outer])

# Calculate percentages of Inner and Outer Follicle cells within the cluster
prop_inner = inner_cells / total_cells_in_cluster * 100
prop_outer = outer_cells / total_cells_in_cluster * 100


# Plotting
plt.figure(figsize=(8, 8), dpi=300)

# Plot other cells in light gray
plt.scatter(
    region_data_filtered.loc[~(mask_inner | mask_outer), 'x'],
    region_data_filtered.loc[~(mask_inner | mask_outer), 'y'],
    c='lightgray', s=2, alpha=0.3, label='Other Cells'
)

# Plot Inner Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_inner, 'x'],
    region_data_filtered.loc[mask_inner, 'y'],
    c='teal', s=2, alpha=0.8, label=f'Inner Follicle ({prop_inner:.2f}%)'
)

# Plot Outer Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_outer, 'x'],
    region_data_filtered.loc[mask_outer, 'y'],
    c='orange', s=2, alpha=0.8, label=f'Outer Follicle ({prop_outer:.2f}%)'
)

# Remove axes
plt.axis('off')

# Title styling
plt.title(f'Cluster {target_prob} in Inner/Outer Follicle\n{target_region}',
          fontsize=25, pad=20)

# Legend styling
plt.legend(
    fontsize=20,
    title_fontsize=20,
    markerscale=10
)

plt.tight_layout()
plt.show()

In [ ]:
# Define your filter
target_prob = 5
neigh_inner = 'Inner Follicle'
neigh_outer = 'Outer Follicle'
target_region = 'B006_Descending - Sigmoid'

# Filter for the specific region of interest
region_data = total_df[total_df['unique_region'] == target_region]

# Filter for the x and y range
region_data_filtered = region_data[(region_data['x'] >= 3000) & (region_data['x'] <= 5000) & 
                                   (region_data['y'] >= 2500) & (region_data['y'] <= 4500)]

# Masks for matching cells in Inner and Outer Follicle neighborhoods
mask_inner = (
    (region_data_filtered['Neighborhood'] == neigh_inner) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)
mask_outer = (
    (region_data_filtered['Neighborhood'] == neigh_outer) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)

# Calculate the total number of cells in Inner and Outer Follicle neighborhoods
total_cells_in_cluster = len(region_data_filtered[mask_inner | mask_outer])

# Calculate the number of Inner and Outer Follicle cells in the cluster
inner_cells = len(region_data_filtered[mask_inner])
outer_cells = len(region_data_filtered[mask_outer])

# Calculate percentages of Inner and Outer Follicle cells within the cluster
prop_inner = inner_cells / total_cells_in_cluster * 100
prop_outer = outer_cells / total_cells_in_cluster * 100


# Plotting
plt.figure(figsize=(8, 8), dpi=300)

# Plot other cells in light gray
plt.scatter(
    region_data_filtered.loc[~(mask_inner | mask_outer), 'x'],
    region_data_filtered.loc[~(mask_inner | mask_outer), 'y'],
    c='lightgray', s=2, alpha=0.3, label='Other Cells'
)

# Plot Inner Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_inner, 'x'],
    region_data_filtered.loc[mask_inner, 'y'],
    c='teal', s=2, alpha=0.8, label=f'Inner Follicle ({prop_inner:.2f}%)'
)

# Plot Outer Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_outer, 'x'],
    region_data_filtered.loc[mask_outer, 'y'],
    c='orange', s=2, alpha=0.8, label=f'Outer Follicle ({prop_outer:.2f}%)'
)

# Remove axes
plt.axis('off')

# Title styling
plt.title(f'Cluster {target_prob} in Inner/Outer Follicle\n{target_region}',
          fontsize=25, pad=20)

# Legend styling
plt.legend(
    fontsize=20,
    title_fontsize=20,
    markerscale=10
)

plt.tight_layout()
plt.show()


In [ ]:
# Define your filter
target_prob = 8
neigh_inner = 'Inner Follicle'
neigh_outer = 'Outer Follicle'
target_region = 'B006_Descending - Sigmoid'

# Filter for the specific region of interest
region_data = total_df[total_df['unique_region'] == target_region]

# Filter for the x and y range
region_data_filtered = region_data[(region_data['x'] >= 3000) & (region_data['x'] <= 5000) & 
                                   (region_data['y'] >= 2500) & (region_data['y'] <= 4500)]

# Masks for matching cells in Inner and Outer Follicle neighborhoods
mask_inner = (
    (region_data_filtered['Neighborhood'] == neigh_inner) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)
mask_outer = (
    (region_data_filtered['Neighborhood'] == neigh_outer) & 
    (region_data_filtered['Probability_Bin_Cluster'] == target_prob)
)

# Calculate the total number of cells in Inner and Outer Follicle neighborhoods
total_cells_in_cluster = len(region_data_filtered[mask_inner | mask_outer])

# Calculate the number of Inner and Outer Follicle cells in the cluster
inner_cells = len(region_data_filtered[mask_inner])
outer_cells = len(region_data_filtered[mask_outer])

# Calculate percentages of Inner and Outer Follicle cells within the cluster
prop_inner = inner_cells / total_cells_in_cluster * 100
prop_outer = outer_cells / total_cells_in_cluster * 100


# Plotting
plt.figure(figsize=(8, 8), dpi=300)

# Plot other cells in light gray
plt.scatter(
    region_data_filtered.loc[~(mask_inner | mask_outer), 'x'],
    region_data_filtered.loc[~(mask_inner | mask_outer), 'y'],
    c='lightgray', s=2, alpha=0.3, label='Other Cells'
)

# Plot Inner Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_inner, 'x'],
    region_data_filtered.loc[mask_inner, 'y'],
    c='teal', s=2, alpha=0.8, label=f'Inner Follicle ({prop_inner:.2f}%)'
)

# Plot Outer Follicle cells
plt.scatter(
    region_data_filtered.loc[mask_outer, 'x'],
    region_data_filtered.loc[mask_outer, 'y'],
    c='orange', s=2, alpha=0.8, label=f'Outer Follicle ({prop_outer:.2f}%)'
)

# Remove axes
plt.axis('off')

# Title styling
plt.title(f'Cluster {target_prob} in Inner/Outer Follicle\n{target_region}',
          fontsize=25, pad=20)

# Legend styling
plt.legend(
    fontsize=20,
    title_fontsize=20,
    markerscale=10
)

plt.tight_layout()
plt.show()


Plot how the percentage of inner and outer follicle change as you move out this gradient for all unique regions that contain both inner and outer follicle regions.

In [ ]:
import matplotlib.pyplot as plt

# Define the cluster order as specified
cluster_order = ['3', '6', '1', '5', '8']

# Loop through each region
for region in total_df['unique_region'].unique():
    region_data = total_df[total_df['unique_region'] == region]

    # Initialize lists to store the percentages for each cluster
    inner_percentages = []
    outer_percentages = []

    # Flag to check if the region has both Inner and Outer Follicle cells for all clusters
    has_both_neigh_in_all_clusters = True

    # Loop through each cluster in the order specified
    for cluster in cluster_order:
        # Masks for matching cells for Inner and Outer Follicle
        mask_inner = (
            (region_data['Neighborhood'] == 'Inner Follicle') &
            (region_data['Probability_Bin_Cluster'] == int(cluster))
        )
        mask_outer = (
            (region_data['Neighborhood'] == 'Outer Follicle') &
            (region_data['Probability_Bin_Cluster'] == int(cluster))
        )

        # Check if the region has cells from both neighborhoods in this cluster
        if not (mask_inner.any() and mask_outer.any()):
            has_both_neigh_in_all_clusters = False
            break  # No need to check further if both neighborhoods are missing for this cluster

        # Calculate the total number of cells in Inner and Outer Follicle neighborhoods for this cluster
        total_cells_in_cluster = len(region_data[mask_inner | mask_outer])

        # Calculate the number of Inner and Outer Follicle cells in the cluster
        inner_cells = len(region_data[mask_inner])
        outer_cells = len(region_data[mask_outer])

        # Calculate the percentages of Inner and Outer Follicle cells relative to total cells in the cluster
        total_cells_in_cluster = max(total_cells_in_cluster, 1)  # Avoid division by zero
        inner_percentage = (inner_cells / total_cells_in_cluster) * 100
        outer_percentage = (outer_cells / total_cells_in_cluster) * 100

        # Append to the lists
        inner_percentages.append(inner_percentage)
        outer_percentages.append(outer_percentage)

   # Only plot if the region has both Inner and Outer Follicle cells in all clusters
    if has_both_neigh_in_all_clusters:
        # Adjust figure size based on larger font sizes
        figsize = (len(cluster_order) * 2, 8)  # Increase width for larger font sizes

        # Plotting the percentages as a bar graph (no lines connecting the points)
        plt.figure(figsize=figsize, dpi=300)

        # Plot Inner Follicle percentage
        plt.bar(cluster_order[:len(inner_percentages)], inner_percentages, label='Inner Follicle', color='teal', width=0.4, align='center')

        # Plot Outer Follicle percentage
        plt.bar(cluster_order[:len(outer_percentages)], outer_percentages, label='Outer Follicle', color='orange', width=0.4, align='edge')

        # Add labels, title, and legend with larger font sizes
        title_text = f'Change in Inner and Outer Follicle Percentages\n- {region}'
        plt.title(title_text, fontsize=35)
        plt.xlabel('Cluster', fontsize=35)
        plt.ylabel('Percentage of Cells (%)', fontsize=35)
        plt.xticks(cluster_order[:len(inner_percentages)], fontsize=35)  # Correct x-axis as categorical
        plt.legend(fontsize=35)

        # Ensure the y-axis is between 0 and 100
        plt.ylim(0, 100)

        plt.tight_layout()

        # Show the plot
        plt.show()

Overall, this analysis shows that the mixture model framework can be used to identify and quantify gradients in follicle structures.